# Multi-Fraction Experiment (10%, 20%, 30%, 60%)

In [1]:
# ==========================================================
# Strict reproducibility setup & Imports
# ==========================================================
import os
import time
import random
import warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_squared_error, r2_score, mean_absolute_error, mean_absolute_percentage_error,
    accuracy_score, precision_score, recall_score, f1_score
)
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.svm import SVR, SVC
from sklearn.multioutput import MultiOutputRegressor
from IPython.display import display, HTML

warnings.filterwarnings('ignore')

SEED = 42

os.environ['PYTHONHASHSEED'] = str(SEED)
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False

try:
    torch.use_deterministic_algorithms(True)
    print('Deterministic mode enabled successfully.')
except Exception as exc:
    print(f'Warning: full deterministic mode could not be enabled: {exc}')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')


Deterministic mode enabled successfully.
Using device: cpu


In [2]:
# ==========================================================
# Model definitions + physics loss
# ==========================================================
class PINN(nn.Module):
    def __init__(self, input_dim, hidden_dims, n_reg_outputs, n_classes, dropout_rate):
        super().__init__()
        layers = []
        prev_dim = input_dim
        for hidden_dim in hidden_dims:
            layers.append(nn.Linear(prev_dim, hidden_dim))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout_rate))
            prev_dim = hidden_dim
        self.backbone = nn.Sequential(*layers)
        self.regression_head = nn.Linear(prev_dim, n_reg_outputs)
        self.classification_head = nn.Linear(prev_dim, n_classes)

    def forward(self, x):
        shared = self.backbone(x)
        return self.regression_head(shared), self.classification_head(shared)

class StandardMLP(nn.Module):
    def __init__(self, input_dim, hidden_dims, n_reg_outputs, n_classes, dropout_rate):
        super().__init__()
        layers = []
        prev_dim = input_dim
        for hidden_dim in hidden_dims:
            layers.append(nn.Linear(prev_dim, hidden_dim))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout_rate))
            prev_dim = hidden_dim
        self.backbone = nn.Sequential(*layers)
        self.regression_head = nn.Linear(prev_dim, n_reg_outputs)
        self.classification_head = nn.Linear(prev_dim, n_classes)

    def forward(self, x):
        shared = self.backbone(x)
        return self.regression_head(shared), self.classification_head(shared)

def physics_loss_normalized(
    x_tensor_scaled,
    y_pred_reg_scaled,
    scaler_X_obj,
    scaler_y_reg_obj,
    feature_names,
    regression_target_names,
):
    dev = x_tensor_scaled.device
    x_mean = torch.tensor(scaler_X_obj.mean_, dtype=torch.float32, device=dev)
    x_scale = torch.tensor(scaler_X_obj.scale_, dtype=torch.float32, device=dev)
    y_mean = torch.tensor(scaler_y_reg_obj.mean_, dtype=torch.float32, device=dev)
    y_scale = torch.tensor(scaler_y_reg_obj.scale_, dtype=torch.float32, device=dev)

    x_unscaled = x_tensor_scaled * x_scale + x_mean
    y_pred_unscaled = y_pred_reg_scaled * y_scale + y_mean

    fmap = {name: x_unscaled[:, i] for i, name in enumerate(feature_names)}
    W12, W34, W6 = fmap['W12(um)'], fmap['W34(um)'], fmap['W6(um)']
    Idc, L, Cc = fmap['Idc(uA)'], fmap['Length(um)'], fmap['CC(pF)']

    gain_pred = y_pred_unscaled[:, regression_target_names.index('Gain(dB)')]
    bw_pred = y_pred_unscaled[:, regression_target_names.index('Bandwidth(Hz)')]
    gbw_pred = y_pred_unscaled[:, regression_target_names.index('GBW(MHz)')]
    pm_pred = y_pred_unscaled[:, regression_target_names.index('PM(degree)')]
    sr_pred = y_pred_unscaled[:, regression_target_names.index('SlewRate (V/us)')]
    power_pred = y_pred_unscaled[:, regression_target_names.index('Power(uW)')]
    cmrr_pred = y_pred_unscaled[:, regression_target_names.index('CMRR(dB)')]

    u_nCox, u_pCox = 343.98, 107.1
    lambda_n, lambda_p = 0.1, 0.2
    VDD = 1.8
    Id1_amps, Cc_farads = (Idc / 2) * 1e-6, Cc * 1e-12

    gm12 = torch.sqrt(2 * u_nCox * (W12 / L) * (Idc / 2)) * 1e-6
    gm34 = torch.sqrt(2 * u_pCox * (W34 / L) * (Idc / 2)) * 1e-6
    gm6 = torch.sqrt(2 * u_nCox * (W6 / L) * Idc) * 1e-6
    gm_avg = (gm12 + gm34 + gm6) / 3
    ro12 = (1 / (lambda_n * (Idc / 2))) * 1e6
    ro34 = (1 / (lambda_p * (Idc / 2))) * 1e6
    ro6 = (1 / (lambda_n * Idc)) * 1e6
    ro7 = (1 / (lambda_n * Idc)) * 1e6

    gain_calc_db = 20 * torch.log10(((gm12 * (ro12 * ro34) / (ro12 + ro34)) * (gm6 * (ro6 * ro7) / (ro6 + ro7)) * 0.033) + 1e-8)
    C1, C2, C3 = 11.28, 0.133, 1e-6
    gain_linear = 10 ** (gain_pred / 20.0)
    pm_correction = (C1 - (C2 * pm_pred))
    gbw_calc_mhz = gain_linear * bw_pred * pm_correction * C3

    sr_calc_vus = ((Id1_amps / Cc_farads) / 1e6) * 2.88
    power_calc_uw = (VDD * Idc) * 9.17
    cmrr_calc_db = 20 * torch.log10(((gm_avg / 1e-6) * 2.85) + 1e-8)

    loss_gain = F.mse_loss(gain_pred, gain_calc_db) / 6400.0
    loss_gbw = F.mse_loss(gbw_pred, gbw_calc_mhz) / 10000.0
    loss_sr = F.mse_loss(sr_pred, sr_calc_vus) / 400.0
    loss_power = F.mse_loss(power_pred, power_calc_uw) / 9000000.0
    loss_cmrr = F.mse_loss(cmrr_pred, cmrr_calc_db) / 10000.0

    return loss_gain + loss_gbw + loss_sr + loss_power + loss_cmrr

def train_dl_model(model, train_loader, val_loader, scaler_X_obj, scaler_y_reg_obj, feat_cols, reg_cols, epochs=50, is_pinn=False):
    optimizer = optim.Adam(model.parameters(), lr=0.0008657)
    crit_reg = nn.MSELoss()
    crit_class = nn.CrossEntropyLoss()

    alpha_max = 0.58 if is_pinn else 0.0
    class_w = 4.30

    for epoch in range(1, epochs + 1):
        model.train()
        alpha = min(alpha_max, alpha_max * (epoch / 50)) if epoch < 50 else alpha_max

        for xb, yrb, ycb in train_loader:
            if is_pinn:
                xb.requires_grad_(True)
            optimizer.zero_grad()
            pr, pc = model(xb)

            loss_sup = crit_reg(pr, yrb) + (class_w * crit_class(pc, ycb))
            loss_phys = 0.0
            if is_pinn:
                loss_phys = physics_loss_normalized(
                    xb, pr, scaler_X_obj, scaler_y_reg_obj, feat_cols, reg_cols
                )

            loss = (1 - alpha) * loss_sup + alpha * loss_phys if is_pinn else loss_sup
            loss.backward()
            optimizer.step()

    model.eval()
    return model


In [3]:
# ==========================================================
# 3) Experiment Loop
# ==========================================================
file_path = '../Data/FINAL_4CLASSES.csv'
df_full = pd.read_csv(file_path, encoding='utf-8', engine='python')

column_mapping = {
    'Gain(db)': 'Gain(dB)', 'Gain': 'Gain(dB)', 'gain': 'Gain(dB)',
    'Bandwidth': 'Bandwidth(Hz)', 'bandwidth': 'Bandwidth(Hz)',
    'GBW': 'GBW(MHz)', 'gbw': 'GBW(MHz)',
    'Power': 'Power(uW)', 'power': 'Power(uW)',
    'PM': 'PM(degree)', 'PhaseMargin': 'PM(degree)',
    'GM': 'GM(dB)',
    'PSRR': 'PSRR(dB)',
    'SlewRate': 'SlewRate (V/us)', 'SlewRate(V/µs)': 'SlewRate (V/us)',
    'CMRR': 'CMRR(dB)',
    'class': 'Class', 'CLASS': 'Class'
}
df_full.rename(columns={k: v for k, v in column_mapping.items() if k in df_full.columns}, inplace=True)

df_full['Idc(uA)'] = 130.0
df_full['Length(um)'] = 0.18
df_full['CL(pF)'] = 10.0
df_full['CC(pF)'] = 55.0

FEATURE_COLUMNS = [
    'Temperature(°)', 'W12(um)', 'W34(um)', 'W58(um)', 'W6(um)', 'W7(um)',
    'Idc(uA)', 'Length(um)', 'CC(pF)', 'CL(pF)'
]
REGRESSION_TARGETS = [
    'Gain(dB)', 'Bandwidth(Hz)', 'GBW(MHz)', 'Power(uW)', 'PM(degree)',
    'GM(dB)', 'PSRR(dB)', 'SlewRate (V/us)', 'CMRR(dB)'
]
CLASSIFICATION_TARGET = 'Class'

FRACTIONS = [0.1, 0.2, 0.3, 0.6]
consol_results = []

C1, C2, C3 = 11.28, 0.133, 1e-6

for frac in FRACTIONS:
    print(f"\n--- Training on {frac*100:.0f}% of Dataset ---")
    if 'Class' in df_full.columns:
        df = df_full.groupby('Class', group_keys=False).apply(lambda x: x.sample(frac=frac, random_state=42))
    else:
        df = df_full.sample(frac=frac, random_state=42)
    df = df.reset_index(drop=True)
    
    X_raw = df[FEATURE_COLUMNS].fillna(df[FEATURE_COLUMNS].mean())
    y_reg_raw = df[REGRESSION_TARGETS].fillna(df[REGRESSION_TARGETS].mean())
    y_class_raw = df[CLASSIFICATION_TARGET].fillna(df[CLASSIFICATION_TARGET].mode()[0])

    scaler_X = StandardScaler()
    X_scaled = scaler_X.fit_transform(X_raw)

    scaler_y_reg = StandardScaler()
    y_reg_scaled = scaler_y_reg.fit_transform(y_reg_raw)

    le = LabelEncoder()
    y_class_labels = le.fit_transform(y_class_raw)
    n_classes = len(le.classes_)

    X_train_fold, X_test_fold, y_reg_train_fold, y_reg_test_fold, y_class_train_fold, y_class_test_fold = train_test_split(
        X_scaled, y_reg_scaled, y_class_labels, test_size=0.2, random_state=42, stratify=y_class_labels
    )
    
    X_train_t = torch.tensor(X_train_fold, dtype=torch.float32, device=device)
    y_reg_train_t = torch.tensor(y_reg_train_fold, dtype=torch.float32, device=device)
    y_class_train_t = torch.tensor(y_class_train_fold, dtype=torch.long, device=device)

    X_test_t = torch.tensor(X_test_fold, dtype=torch.float32, device=device)
    y_reg_test_t = torch.tensor(y_reg_test_fold, dtype=torch.float32, device=device)
    y_class_test_t = torch.tensor(y_class_test_fold, dtype=torch.long, device=device)

    dl_train = DataLoader(TensorDataset(X_train_t, y_reg_train_t, y_class_train_t), batch_size=64, shuffle=True)
    dl_test = DataLoader(TensorDataset(X_test_t, y_reg_test_t, y_class_test_t), batch_size=64, shuffle=False)

    # PINN
    pinn = PINN(X_train_fold.shape[1], [128, 128, 128, 128], len(REGRESSION_TARGETS), n_classes, 0.047).to(device)
    pinn = train_dl_model(pinn, dl_train, dl_test, scaler_X, scaler_y_reg, FEATURE_COLUMNS, REGRESSION_TARGETS, epochs=50, is_pinn=True)
    pr, pc = pinn(X_test_t)
    pred_reg_pinn = scaler_y_reg.inverse_transform(pr.cpu().detach().numpy())

    # Standard MLP
    mlp = StandardMLP(X_train_fold.shape[1], [128, 128, 128, 128], len(REGRESSION_TARGETS), n_classes, 0.047).to(device)
    mlp = train_dl_model(mlp, dl_train, dl_test, scaler_X, scaler_y_reg, FEATURE_COLUMNS, REGRESSION_TARGETS, epochs=50, is_pinn=False)
    pr, pc = mlp(X_test_t)
    pred_reg_mlp = scaler_y_reg.inverse_transform(pr.cpu().detach().numpy())

    # RF
    rf_reg = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
    y_reg_train_raw = scaler_y_reg.inverse_transform(y_reg_train_fold)
    rf_reg.fit(X_train_fold, y_reg_train_raw)
    pred_reg_rf = rf_reg.predict(X_test_fold)

    # SVM
    svm_reg = MultiOutputRegressor(SVR(kernel='rbf'))
    svm_reg.fit(X_train_fold, y_reg_train_fold)
    pred_reg_svm = scaler_y_reg.inverse_transform(svm_reg.predict(X_test_fold))

    y_true_reg = scaler_y_reg.inverse_transform(y_reg_test_fold)

    preds = {
        'PINN': pred_reg_pinn,
        'Standard MLP': pred_reg_mlp,
        'Random Forest': pred_reg_rf,
        'SVM': pred_reg_svm,
    }

    # Evaluate
    for model_name, pred_reg in preds.items():
        # Overall Target R2 & MAE
        overall_r2 = r2_score(y_true_reg, pred_reg)
        overall_mae = mean_absolute_error(y_true_reg, pred_reg)
        
        # Physics Consistency
        idx_gain = REGRESSION_TARGETS.index('Gain(dB)')
        idx_bw = REGRESSION_TARGETS.index('Bandwidth(Hz)')
        idx_gbw = REGRESSION_TARGETS.index('GBW(MHz)')
        idx_pm = REGRESSION_TARGETS.index('PM(degree)')

        p_gain = pred_reg[:, idx_gain]
        p_bw = pred_reg[:, idx_bw]
        p_gbw = pred_reg[:, idx_gbw]
        p_pm = pred_reg[:, idx_pm]

        gain_lin = 10 ** (p_gain / 20.0)
        pm_corr = (C1 - (C2 * p_pm))
        gbw_calc = gain_lin * p_bw * pm_corr * C3

        consist_r2 = r2_score(p_gbw, gbw_calc)

        consol_results.append({
            'Data_Fraction (%)': int(frac*100),
            'Model': model_name,
            'Overall_R2': overall_r2,
            'Overall_MAE': overall_mae,
            'Physics_Consistency_R2': consist_r2
        })

print('\nDone training all fractions.')



--- Training on 10% of Dataset ---

--- Training on 20% of Dataset ---

--- Training on 30% of Dataset ---

--- Training on 60% of Dataset ---

Done training all fractions.


In [4]:
# ==========================================================
# 4) Consolidated Table Results
# ==========================================================
import pandas as pd

df_results = pd.DataFrame(consol_results)

# Sort by Fraction (Ascending) and Overall R2 (Descending)
df_sorted = df_results.sort_values(by=['Data_Fraction (%)', 'Overall_R2'], ascending=[True, False]).reset_index(drop=True)

# Format floats for readability
df_display = df_sorted.copy()
df_display['Overall_R2'] = df_display['Overall_R2'].apply(lambda x: f"{x:.4f}")
df_display['Overall_MAE'] = df_display['Overall_MAE'].apply(lambda x: f"{x:.4f}")
df_display['Physics_Consistency_R2'] = df_display['Physics_Consistency_R2'].apply(lambda x: f"{x:.4f}")

# Display HTML tables directly inside jupyter
print("--- SUMMARY OF MODEL PERFORMANCES AT DIFFERENT DATASET SIZES ---")
display(HTML(df_display.to_html(index=False)))

print("\nWe can clear see how models perform under low data situations:")
for frac in df_sorted['Data_Fraction (%)'].unique():
    best_model = df_sorted[df_sorted['Data_Fraction (%)'] == frac].iloc[0]['Model']
    print(f"At {frac}% Data: Best strictly by R2 is -> {best_model}")


--- SUMMARY OF MODEL PERFORMANCES AT DIFFERENT DATASET SIZES ---


Data_Fraction (%),Model,Overall_R2,Overall_MAE,Physics_Consistency_R2
10,Random Forest,0.9960,56.7747,0.7239
10,SVM,0.9907,397.7335,0.8796
10,Standard MLP,0.9813,528.9803,0.8217
10,PINN,0.9770,510.9558,0.9597
20,Random Forest,0.9982,42.7508,0.6788
20,SVM,0.9908,404.1754,0.8537
20,Standard MLP,0.9891,394.2153,0.8494
20,PINN,0.9847,431.4797,0.9555
30,Random Forest,0.9991,25.4608,0.7128
30,Standard MLP,0.9927,206.3369,0.7685



We can clear see how models perform under low data situations:
At 10% Data: Best strictly by R2 is -> Random Forest
At 20% Data: Best strictly by R2 is -> Random Forest
At 30% Data: Best strictly by R2 is -> Random Forest
At 60% Data: Best strictly by R2 is -> Random Forest
